# Lab 3: Text Preparation

**Business question:** Which cleaning changes keep the words a service decision needs?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | 24 customer messages and six test cases | about 35 min | Run outputs, saved files and the checkpoint |
| 2 Practice | CFPB complaint extract | about 30–45 min | Three exercises and the interpretation |
| 3 Challenge | Both | optional | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

## Part 1 Guided core

You continue the service review case from Modules 1 and 2. You keep the original text, the message IDs and the Module 2 partition. You write one small cleaning function, compare two cleaned versions and save a table of what changed.

### 1.1 Load the tools

Select the `ban6403` kernel in VS Code, then run the cell below. `pathlib` and `unicodedata` come with Python. No output means it worked. If the import fails, check that `ban6403` is selected and pandas is installed in that environment. The Module 0 readiness notebook shows what to fix.

In [ ]:
# Path builds file paths; unicodedata handles character forms; pandas handles tables.
from pathlib import Path
import unicodedata
import pandas as pd

### 1.2 Point to the data folder

The notebook runs from the `notebooks` folder, so the data folder is one level up: `../data`. All later file paths start from `DATA`. This cell shows no output.

In [ ]:
# ".." means one folder up: the data folder next to notebooks/.
DATA = Path("../data")

### 1.3 Read the customer messages

We read the 24 feedback messages. We never overwrite this file. Expect `(24, 5)`: 24 messages and 5 columns.

In [ ]:
# Read the 24 messages; shape gives (rows, columns).
feedback = pd.read_csv(DATA / "raw/customer_feedback_m01.csv")
feedback.shape

### 1.4 Load the partition from Module 2

A partition is a group of records with one job, such as development or testing. We reuse the Module 2 file instead of making a new split. Expect 13 development, 10 protected and 1 excluded record.

In [ ]:
# Read the Module 2 partition and count the records in each partition.
partition = pd.read_csv(DATA / "partitions/m02_partition.csv")
partition["partition"].value_counts()

### 1.5 Select the development IDs

We choose cleaning rules using development messages and our own test examples only. The protected messages stay unread, so they remain a fair test later. The table shows the first five rows of the manifest: the file that lists each record, its group and its partition.

In [ ]:
# .eq("development") compares each value with "development": True for development rows.
# .copy() makes a separate table, so later changes do not touch partition.
dev_manifest = partition.loc[partition["partition"].eq("development")].copy()
dev_ids = dev_manifest["feedback_id"]
dev_manifest.head()

### 1.6 Get the development messages

The message ID links each text to the manifest. Expect `(13, 1)`. FB005 and FB020 have exactly the same text, and both are included.

In [ ]:
# Use the message IDs as row labels, then pick the development IDs.
messages = feedback.set_index("feedback_id")
dev = messages.loc[dev_ids, ["text"]].copy()
dev.shape

### 1.7 Read FB005 closely

Read the whole message. Under the Module 2 policy, "No action is required" means this message needs no review. The word *refund* alone cannot tell a new request from a solved case. We will check that this phrase survives cleaning.

In [ ]:
# Print the full text of FB005.
print(dev.loc["FB005", "text"])

### 1.8 Write one small cleaning function

`prepare_light` does two things:

1. **NFC.** Unicode is the standard computers use to store characters. Some characters can be stored in two ways: *é* can be one character, or a plain *e* followed by an accent mark. NFC (a Unicode normalization form) stores both the same way.
2. **Single spaces.** `split()` breaks the text on any whitespace: spaces, tabs and line breaks. `" ".join(...)` puts the pieces back together with one space.

The function keeps letter case, punctuation, numbers and words like *no* and *not*. It does not assign labels, remove HTML or fix spelling. We call this rule set `light_v1`. Joining line breaks is fine for short messages, but it can break the layout of longer documents or tables.

Running the cell only defines the function, so you see no output.

In [ ]:
def prepare_light(text):
    """Take one text and return it in NFC form with single spaces."""
    # Store characters such as é in one standard way.
    normalized = unicodedata.normalize("NFC", text)
    # split() breaks on any whitespace; " ".join puts the words back with one space.
    return " ".join(normalized.split())

### 1.9 Try the function on one string

`repr` shows hidden characters such as extra spaces. Expect the spaces at the start, the end and between words to shrink to single spaces. The words "No action is required." all stay.

In [ ]:
example = "  No   action is required.  "
# repr() shows the quotes and every space, so you can see what changed.
print("Before:", repr(example))
print("After: ", repr(prepare_light(example)))

### 1.10 Load the test cases

The six test cases (TC01–TC06) are short strings we wrote to test cleaning rules. They are separate from the 24 customer messages, and we do not use them to evaluate anything.

`repr` makes hidden characters visible. TC02 contains a tab, shown as `\t`. TC03 looks like "Café", but its *é* is stored as a plain *e* followed by a separate accent mark.

In [ ]:
tests = pd.read_csv(DATA / "raw/m03_preparation_test_cases.csv")
# Use the test IDs (TC01 to TC06) as row labels.
tests = tests.set_index("test_id")
print(repr(tests.loc["TC02", "text"]))
print(repr(tests.loc["TC03", "text"]))

### 1.11 Clean the test cases

`map` applies the same function to every test string. Check each row:

- TC01 and TC02 lose their extra spaces and tab.
- TC03 now stores *é* as one character. It looks the same on screen.
- TC04 keeps its wide letters `ＡＢ１２`, because NFC does not change them.
- TC06 loses its line break. That is a limit for text where layout matters.

In [ ]:
# map() runs prepare_light on every text and returns the results as a new column.
tests["light_text"] = tests["text"].map(prepare_light)
tests[["text", "light_text", "purpose"]]

### 1.12 Add a lowercase version

Lowercase can help when you count words, because *Refund* and *refund* become one word. We keep it in a separate column called `lexical_candidate`, so the choice stays visible. (*Lexical* means based on words.)

Compare three rows. TC01 still has its negation. TC02 keeps the amount, but `AB-12` becomes `ab-12`. In TC05, `Ab-12` and `AB-12` become the same. Whether that is safe depends on the product catalog.

In [ ]:
# .str applies a string method to every row; lower() makes all letters small.
tests["lexical_candidate"] = tests["light_text"].str.lower()
tests.loc[["TC01", "TC02", "TC05"], ["light_text", "lexical_candidate"]]

### 1.13 See a harmful deletion

This cell deletes "No" on purpose to show the risk. "No action is required." becomes "action is required.", which says the opposite. We never apply this rule to the messages.

In [ ]:
resolved = tests.loc["TC01", "light_text"]
print("Preserved:", resolved)
# replace(old, new, 1) replaces only the first "No ".
print("Harmful: ", resolved.replace("No ", "", 1))

### 1.14 Look at token boundaries

A token is one piece of text, such as a word. Splitting on spaces keeps `don't`, `AB-12` and `$49.99` whole. Later tools have their own tokenizers and may split these differently. Module 4 shows one.

In [ ]:
token_example = "don't refund AB-12 for $49.99"
# With no argument, split() separates text on runs of whitespace, including spaces, tabs and line breaks.
token_example.split()

### 1.15 Clean the development messages

Now apply the function to the 13 development messages. The new `light_text` column sits beside the original `text`. `light_changed` is `True` when the cleaned text differs from the original.

Expect 13 `False`. These messages already use normal spaces and characters, so zero changes is a valid result. The test cases showed what the function does when there is something to fix.

In [ ]:
dev["light_text"] = dev["text"].map(prepare_light)
# ne() means "not equal": True where cleaning changed the text.
dev["light_changed"] = dev["text"].ne(dev["light_text"])
dev["light_changed"].value_counts()

### 1.16 Keep lowercase as a second view

Expect 13 `True`, because every message starts with a capital letter. This view still keeps all words and punctuation. It is a candidate for word counts in Module 4. We have not yet tested whether it helps.

In [ ]:
# A second view: the light text in lower case.
dev["lexical_candidate"] = dev["light_text"].str.lower()
dev["lexical_changed"] = dev["light_text"].ne(dev["lexical_candidate"])
dev["lexical_changed"].value_counts()

### 1.17 Compare two messages

FB005 is solved. FB008 asks staff to check a double charge. Check that "No action is required" and "Please check the bill" survive in both cleaned views.

In [ ]:
# Each print shows the original and both cleaned views of one message.
print(dev.loc["FB005", ["text", "light_text", "lexical_candidate"]])
print(dev.loc["FB008", ["text", "light_text", "lexical_candidate"]])

### 1.18 Build the audit table

An audit table puts each ID, group and partition next to the original text, both cleaned views and the change flags. The `policy` column names the rules, so another analyst can repeat them. The cell shows the first five rows.

In [ ]:
# reset_index() turns the feedback_id row labels back into a column,
# so merge can match each message to its manifest row by feedback_id.
audit = dev_manifest.merge(dev.reset_index(), on="feedback_id")
# The same text in every row: the name of the rules we used.
audit["policy"] = "light_v1; lexical_candidate adds lowercase"
audit[["feedback_id", "group_id", "partition", "light_changed", "lexical_changed"]].head()

### 1.19 Check the repeated text

FB005 and FB020 keep their own IDs. They share one group ID (the label we gave to copies of the same text) in the development partition. A shared group does not mean the same customer. Cleaning gives no reason to delete one copy or move it to another partition.

In [ ]:
# Rows of the repeated-text group, with four columns.
in_group = audit["group_id"].eq("exact_refund_text")
audit.loc[in_group, ["feedback_id", "group_id", "partition", "light_text"]]

### 1.20 Record the excluded message

FB022 has no text. It stays in a separate exclusion record with its reason. We do not guess its text or use its star rating as a label.

In [ ]:
# The excluded record (FB022) and its reason.
excluded = partition.loc[partition["partition"].eq("excluded")].copy()
excluded[["feedback_id", "reason"]]

### 1.21 Save your outputs

The next four cells create the folder `outputs/m03` and write three small CSV files:

- `preparation_audit.csv`: the 13 development messages with both cleaned views
- `excluded_records.csv`: FB022 and its reason
- `test_case_comparison.csv`: the six test cases

No protected text is saved, and the raw data is not changed. After saving the notebook, review and commit the changes in GitHub Desktop.

In [ ]:
OUTPUT_DIR = Path("../outputs/m03")
# parents=True also creates outputs/ if needed; exist_ok=True accepts an existing folder.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# index=False leaves out pandas' row numbers.
audit.to_csv(OUTPUT_DIR / "preparation_audit.csv", index=False)
print(OUTPUT_DIR / "preparation_audit.csv")

In [ ]:
# index=False leaves out pandas' row numbers.
excluded.to_csv(OUTPUT_DIR / "excluded_records.csv", index=False)

In [ ]:
# The test IDs are row labels; index_label names that column in the file.
tests.to_csv(OUTPUT_DIR / "test_case_comparison.csv", index_label="test_id")

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. Your audit shows that the light view changed none of the 13 development messages, while lowercase changed all 13. Why is zero light changes a valid result?
2. Name one cleaning rule you would reject for the review queue. Use FB005 or one test case as evidence.

**Your answer:**

## Part 2 Practice: real complaints

Now apply the same function to real complaints sent to the U.S. Consumer Financial Protection Bureau (CFPB). Each row is one complaint in the consumer's own words. The extract is already split into `train`, `valid` and `test`.

Imagine a complaint unit that wants to count words in order to route each complaint to the right department. Before anyone counts words, you check what each cleaning rule does to real text.

Three things to know about this data:

- CFPB hides personal details with `XXXX`. Dates are hidden as `XX/XX/XXXX` or `XX/XX/year>`; step 2.8 counts both. Amounts appear in braces, such as `{$450.00}`.
- Complaints are much longer than the 24 customer messages and often have line breaks.
- Complaints come from people who chose to file one, so counts describe this dataset, not all customers.

### 2.1 Load the complaint extract

Expect `(10000, 13)`: 10,000 complaints and 13 columns.

In [ ]:
# Read the complaints; shape gives (rows, columns).
complaints = pd.read_csv(DATA / "course/cfpb_complaints.csv")
complaints.shape

### 2.2 Keep the training complaints

We choose cleaning rules on `train` only, just as we used only development messages in Part 1. The `valid` and `test` complaints stay unread. `.copy()` makes `train` a separate table, so new columns do not touch `complaints`. Expect `(6000, 13)`.

In [ ]:
# .copy() makes train a separate table, so new columns do not touch complaints.
train = complaints[complaints["split"].eq("train")].copy()
train.shape

### 2.3 Read one raw complaint

`repr` shows the hidden characters. Look for `\n` (a line break), a double space, `XXXX`, a date written as `XX/XX/year>` and the amount `{$3500.00}`. The `XX/XX/year>` form is a leftover from CFPB's own redaction tool. The consumer did not type it.

In [ ]:
# The narrative of one complaint; iloc[0] takes the single value out of the column.
sample = train.loc[train["complaint_id"].eq(15095159), "narrative"].iloc[0]
print(repr(sample))

### 2.4 Clean the same complaint

The same `prepare_light` function from 1.8 now runs on real text. The line break and the double space become single spaces. Every word, `XXXX` and the amount stay.

In [ ]:
# The same complaint after cleaning; repr() shows every space.
print(repr(prepare_light(sample)))

### 2.5 Clean all training complaints

This adds `light_text` beside the original `narrative` and flags each complaint that changed. Expect about two thirds of the complaints to change. This cell can take a second or two.

In [ ]:
# The same two steps as 1.15, now on 6,000 complaints.
train["light_text"] = train["narrative"].map(prepare_light)
train["light_changed"] = train["narrative"].ne(train["light_text"])
train["light_changed"].value_counts()

### 2.6 What caused the changes?

The function has two steps, so we check them one at a time. First we apply NFC alone. Then we count complaints with a line break or a double space. Expect NFC to change nothing here: every change comes from whitespace.

In [ ]:
# Step 1 alone: NFC without the whitespace step.
nfc_text = train["narrative"].str.normalize("NFC")
print("changed by NFC:", train["narrative"].ne(nfc_text).sum())

# Step 2: count complaints that have whitespace to fix. "\n" is a line break.
print("has a line break:", train["narrative"].str.contains("\n", regex=False).sum())
print("has a double space:", train["narrative"].str.contains("  ", regex=False).sum())

### 2.7 Add the lowercase view

Same as 1.16. Almost every complaint changes, because almost every complaint has a capital letter. The few that do not were written entirely in lowercase.

In [ ]:
# The same lowercase view as in 1.16, now for every training complaint.
train["lower_text"] = train["light_text"].str.lower()
# True where lowercasing changed the text.
train["lower_changed"] = train["light_text"].ne(train["lower_text"])
train["lower_changed"].value_counts()

### 2.8 Count the real noise

How many training complaints contain each kind of redaction or amount? `regex=False` tells pandas to look for the exact characters.

In [ ]:
raw = train["narrative"]
# One count per kind of mark; the dictionary keys become the row labels.
noise = pd.Series({
    "XXXX": raw.str.contains("XXXX", regex=False).sum(),
    "XX/XX/XXXX": raw.str.contains("XX/XX/XXXX", regex=False).sum(),
    "XX/XX/year>": raw.str.contains("XX/XX/year>", regex=False).sum(),
    "amount in braces": raw.str.contains("{$", regex=False).sum(),
})
noise

### 2.9 How long are the complaints?

We count words by splitting `light_text` on spaces. `describe()` summarizes the word counts. Its rows are `count` (complaints), `mean` (average) and `std` (standard deviation, the typical distance from the mean). Then come `min`, the quartiles `25%`, `50%` (the median) and `75%`, and `max`. The 24 customer messages had about ten words each. Compare that with the median and the maximum here.

In [ ]:
# split() makes a list of words; str.len() counts them.
train["word_count"] = train["light_text"].str.split().str.len()
train["word_count"].describe().round(1)

### 2.10 Try a candidate rule: delete redaction tokens

A teammate suggests deleting every token that contains `XX`, so the redactions do not fill up the word counts. The pattern `\S*XX\S*` means "any run of non-space characters that contains XX". After deleting, we collapse the leftover spaces again. The new column sits beside `light_text`; nothing is overwritten.

In [ ]:
# Delete every token that contains XX (regex=True: the pattern is a regular expression).
deleted = train["light_text"].str.replace(r"\S*XX\S*", "", regex=True)
# Deleting leaves double spaces, so collapse the spaces again.
train["no_xxxx_text"] = deleted.str.split().str.join(" ")

train["no_xxxx_words"] = train["no_xxxx_text"].str.split().str.len()
print("words before:", train["word_count"].sum())
print("words after: ", train["no_xxxx_words"].sum())

### 2.11 What did the rule remove?

Compare the sample complaint before and after. The counts got smaller, but the text now reads "dated the account" and "from ( Reference ), the agency representing confirming". The reader can no longer see that dates and names were there.

In [ ]:
in_sample = train["complaint_id"].eq(15095159)
# [:300] keeps the first 300 characters of each version.
print(train.loc[in_sample, "light_text"].iloc[0][:300])
print(train.loc[in_sample, "no_xxxx_text"].iloc[0][:300])

### 2.12 Negation in real complaints

Many default stopword lists remove *no* and *not*. (Stopwords are very common words that some tools delete.) How many training complaints contain the word *not*? We search the lowercase view for *not* as a whole word. In a regular expression, `\b` marks a word boundary: the start or end of the text, a space or a punctuation mark. So *not.* and *(not* count, but *nothing* and *note* do not.

In [ ]:
# Match "not" at word boundaries, including beside punctuation.
has_not = train["lower_text"].str.contains(r"\bnot\b", regex=True)
print("complaints with the word 'not':", has_not.sum(), "of", len(train))

### 2.13 What deleting *not* does

Here is the start of one real complaint, first as written and then with *not* removed. Read both lines. The second line says the accounts are the consumer's own and that the consumer applied for them.

In [ ]:
# The first 300 characters of one complaint, then the same text without "not".
example = train.loc[train["complaint_id"].eq(14531587), "light_text"].iloc[0][:300]
print(example)
print(example.replace(" not ", " "))

### Exercise 1: a placeholder instead of deletion

Instead of deleting redactions (2.10), replace them with a visible placeholder.

1. Create a new column `redacted_text` from `light_text`. Replace every group of four or more X's with `[REDACTED]`. Hint: `.str.replace(r"X{4,}", "[REDACTED]", regex=True)`.
2. Print how many complaints changed (compare `light_text` with `redacted_text`).
3. Print the sample complaint 15095159 in its new form.

This exercise replaces long X runs only. A date such as `XX/XX/XXXX` becomes `XX/XX/[REDACTED]`, because its `XX` parts are too short to match. Replacing whole dates belongs to Challenge D.

Do not change `narrative` or `light_text`.

In [ ]:
# Exercise 1: your code here
# TODO: create train["redacted_text"] from train["light_text"] with .str.replace(r"X{4,}", "[REDACTED]", regex=True).
# TODO: print how many complaints changed (.ne() compares the two columns; .sum() counts True).
# TODO: print the redacted_text of complaint 15095159.

### Exercise 2: when does lowercase lose information?

Find one word whose capital form means something different. One example: *Discover* is a credit card company, while *discover* is an ordinary verb. Another is *US* (the country) and *us*.

1. Count how many complaints contain your word with a capital letter in `light_text`, and how many contain the small-letter form in `light_text`.
2. Count how many contain the small-letter form in `lower_text`.
3. Take one complaint whose original `light_text` uses the small-letter form. Print its `complaint_id` and about 100 characters before and after the word, so you can read how it is used.

If the small-letter count is zero, say so, and choose another word for step 3.

Hint: `r"\bDiscover\b"` matches the whole word only. `\b` marks a word boundary. To find where the word starts in one text, use `re.search(r"\bdiscover\b", text).start()` (add `import re` first). If you think lowercase loses nothing useful here, say so in your interpretation and explain why.

In [ ]:
# Exercise 2: your code here
# TODO: count complaints whose light_text contains your capital word, e.g. r"\bDiscover\b".
# TODO: count complaints whose light_text contains the small-letter form, then do the same in lower_text.
# TODO: if the small-letter count is 0, choose another word.
# TODO: take one light_text with the small-letter form; print its complaint_id and about 100 characters around the word.

### Exercise 3: paragraphs and line breaks

Test case TC06 showed that the light function removes line breaks. Check what that means for real complaints. A blank line (`"\n\n"`) usually starts a new paragraph.

1. Create a column `blank_lines` with the number of `"\n\n"` in each raw `narrative` (use `.str.count`).
2. Print how many complaints have at least one blank line, and the largest number in one complaint.
3. Print how many blank lines are left in `light_text` in total.

In [ ]:
# Exercise 3: your code here
# TODO: create train["blank_lines"] with train["narrative"].str.count("\n\n").
# TODO: print how many complaints have at least one blank line, and the largest number.
# TODO: print the total number of "\n\n" left in light_text.

### Practice interpretation

Write 3–5 sentences, using numbers from your own outputs:

- How many training complaints did the light function change, and what caused those changes? Is that acceptable for counting words?
- Compare deleting redaction tokens (2.10–2.11) with your `[REDACTED]` placeholder. Which would you give the complaint team, and why?
- Which view would you use for reading a complaint, and which for counting words? Use your lowercase example and your paragraph count.
- What does the whitespace split in 1.14 keep together (for example `don't` or `AB-12`), and why does that matter for your task?

**Your answer:**

### Submit

1. Run the whole notebook from the top (**Restart**, then **Run All**) and check that there are no errors.
2. Save the notebook, so the outputs are saved in the file. The saved outputs are what is graded.
3. In GitHub Desktop, write a short summary such as "Lab 3 complete", click **Commit to main**, then **Push origin**.
4. Push before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

If you used an AI tool to explain or debug code, add one line here: the tool, what you used it for, and how you checked the result.

**AI use:**

## Part 3 Challenge (optional, not graded)

### A. Does cleaning twice change anything?

A good cleaning function gives the same result when you run it a second time. This property is called *idempotence*. Without it, the cleaned text would depend on how often someone ran the notebook. Expect 0. Passing this check does not prove that the meaning survived; you still need to read the text.

In [ ]:
# Clean the already cleaned text once more and count the rows that change.
cleaned_twice = audit["light_text"].map(prepare_light)
print("messages changed by a second cleaning:", audit["light_text"].ne(cleaned_twice).sum())

### B. NFC or NFKC?

NFKC is a stronger Unicode form. It also turns look-alike characters, such as the wide letters in TC04, into ordinary ones. Compare the two forms below. Would a product catalog want `ＡＢ１２` and `AB12` to be the same code? Decide before you use NFKC. This cell does not change the audit.

In [ ]:
code_example = tests.loc["TC04", "text"]
# The same text in both Unicode forms.
print("NFC: ", unicodedata.normalize("NFC", code_example))
print("NFKC:", unicodedata.normalize("NFKC", code_example))

### C. Test your own inputs

Good test cases come from risks you can name. Below are two new short messages: one with a double space and one with a line break.

1. Add one more string of your own that tests a risk you care about. Examples: a tab, an accented name or a product code.
2. Then clean all rows with `prepare_light` and add a lowercase column, so your new row gets both views too.
3. Check what each view keeps and loses.

Keep the `check_id` column, so these strings are never confused with customer messages.

In [ ]:
# Two new test strings, each with its own check_id.
my_tests = pd.DataFrame({
    "check_id": ["IN01", "IN02"],
    "text": ["Please send  the missing charger.", "The refund arrived.\nNo action is required."],
})
my_tests

In [ ]:
# Challenge C: your code here
# TODO: add one row of your own first, for example:
# new_row = pd.DataFrame({"check_id": ["IN03"], "text": ["your text here"]})
# my_tests = pd.concat([my_tests, new_row], ignore_index=True)
# TODO: then add a light_text column for all rows with my_tests["text"].map(prepare_light).
# TODO: add a lexical_candidate column with .str.lower(), then show my_tests.

### D. One rule for every redacted date

CFPB dates appear in several forms in `train`: `XX/XX/XXXX`, `XX/XX/year>` and sometimes `XX/XX/2025`. Write one rule that turns all of them into `[DATE]`, then turn the remaining `XXXX` runs into `[REDACTED]`. Count how many complaints contain `[DATE]` afterwards.

Think about order: what happens to `XX/XX/XXXX` if you replace `XXXX` first?

**Pattern key.**

| Pattern | Meaning |
|---|---|
| `XX/XX/` | exactly these characters |
| `(a\|b)` | a group: `a` or `b` (the `\|` means "or") |
| `\d` | one digit, 0 to 9 |
| `\d+` | one or more digits |
| `\d{4}` | exactly four digits |
| `X{4,}` | four or more X's |

For example, `r"XX/XX/(XXXX|year>)"` matches `XX/XX/XXXX` and `XX/XX/year>`. The starter cell begins with this pattern; add the digit form.

In [ ]:
# Challenge D: your code here
# The date pattern so far covers XX/XX/XXXX and XX/XX/year>.
date_pattern = r"XX/XX/(XXXX|year>)"
# TODO: add a third choice inside the group for a year written in digits, e.g. |\d{4}.
# TODO: replace date_pattern in light_text with "[DATE]" (str.replace with regex=True).
# TODO: then replace the remaining runs of four or more X's with "[REDACTED]".
# TODO: count the complaints that contain "[DATE]".

## Further reading

- [Python `unicodedata.normalize`](https://docs.python.org/3/library/unicodedata.html#unicodedata.normalize): NFC and NFKC.
- [Python `str.split`](https://docs.python.org/3/library/stdtypes.html#str.split): how splitting on whitespace works.
- [pandas: working with text data](https://pandas.pydata.org/docs/user_guide/text.html): the `.str` methods used in Practice.

Next, Module 4 turns the cleaned text into word and phrase features: counts and TF-IDF.